In [2]:
%pip install sentence-transformers
%pip install faiss-cpu

  Using cached click-8.4.2-py3-none-any.whl.metadata (2.6 kB)
   ---------------------------------------- 0.0/596.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/596.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/596.4 kB ? eta -:--:--
   ---------------------------------------- 596.4/596.4 kB 2.4 MB/s  0:00:00
   ---------------------------------------- 0.0/11.5 MB ? eta -:--:--
   - -------------------------------------- 0.5/11.5 MB 1.9 MB/s eta 0:00:06
   --- ------------------------------------ 1.0/11.5 MB 2.6 MB/s eta 0:00:05
   ----- ---------------------------------- 1.6/11.5 MB 2.5 MB/s eta 0:00:04
   ------- -------------------------------- 2.1/11.5 MB 2.7 MB/s eta 0:00:04
   ---------- ----------------------------- 2.9/11.5 MB 2.7 MB/s eta 0:00:04
   ------------ --------------------------- 3.7/11.5 MB 2.9 MB/s eta 0:00:03
   --------------- ------------------------ 4.5/11.5 MB 3.0 MB/s eta 0:00:03
   ----------------- ------------------

In [3]:
import pandas as pd

import numpy as np

from sentence_transformers import SentenceTransformer

import faiss

In [4]:
df = pd.read_csv("my_data_q.csv")
df.head()

,quesNo,subjects,lokNo,member,ministry,type,date,sessionNo,supplementaryType,year
0,3008,"Toll Plazas in Dharmapuri, Tamil Nadu ...",18,Shri Mani A,ROAD TRANSPORT AND HIGHWAYS,UNSTARRED,2025-12-18,6,False,2025
1,2960,Programmes to improve Connectivity/Lo...,18,Shri Mani A,RAILWAYS,UNSTARRED,2025-12-17,6,False,2025
2,2570,Schemes to promote Skill Development among Far...,18,Shri Mani A,AGRICULTURE AND FARMERS WELFARE,UNSTARRED,2025-12-16,6,False,2025
3,2401,Contribution of States in Tourism Industry ...,18,Shri Mani A,TOURISM,UNSTARRED,2025-12-15,6,False,2025
4,2226,Pharmacovigilance System for AYUSH Drugs,18,"Shri E T Mohammed Basheer, Shri Mani A",AYUSH,UNSTARRED,2025-12-12,6,False,2025


In [5]:
texts = df["subjects"].fillna("").tolist()

In [6]:
model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

D:\python\anaconda\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\haank\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
embeddings = model.encode(
    texts,
    show_progress_bar=True
)

Batches:   0%|          | 0/1165 [00:00<?, ?it/s]

In [8]:
embeddings = np.array(
    embeddings,
    dtype="float32"
)

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(embeddings)

In [34]:
def search(query, k=5):

    query_vector = model.encode([query]).astype("float32")

    distance, indices = index.search(
        query_vector,
        k
    )

    return df.iloc[
        indices[0]
    ][
        ["subjects","ministry"]
    ]

In [35]:
search(
    "solar energy subsidy"
)

,subjects,ministry
4180,Promotion of Solar Energy,NEW AND RENEWABLE ENERGY
23327,Promotion of Solar Energy,NEW AND RENEWABLE ENERGY
27428,Promotion of Solar Energy,NEW AND RENEWABLE ENERGY
1747,Solar Energy Policy,NEW AND RENEWABLE ENERGY
20748,Solar Energy Policy,NEW AND RENEWABLE ENERGY


In [11]:
search(
    "farmer loan"
)

,subjects,ministry
19960,Agricultural Loan for Small and Marginal Farmers,AGRICULTURE AND FARMERS WELFARE
9915,Loanee Farmers,AGRICULTURE AND FARMERS WELFARE
5862,Schemes for Easy Loans to Farmers,FINANCE
27299,Schemes for Easy Loans to Farmers,FINANCE
26911,Loans Disbursed to Farmers,FINANCE


In [12]:
search(
    "digital payment"
)

,subjects,ministry
30606,Promotion of Digital Payment System,FINANCE
34977,Cyber Security of Digital Payment,FINANCE
26162,Digital Payments for Development Work,PANCHAYATI RAJ
10332,E-commerce Payments,COMMERCE AND INDUSTRY
23035,Lack of Digital Payment System in India Post,COMMUNICATION


In [13]:
search(
    "women safety"
)

,subjects,ministry
10787,Women Safety,HOME AFFAIRS
596,Safety of Women at Workplace,WOMEN AND CHILD DEVELOPMENT
34853,Survey on Women Safety ...,WOMEN AND CHILD DEVELOPMENT
13978,Schemes for Safety of Women,WOMEN AND CHILD DEVELOPMENT
3007,Schemes for Safety of Women,WOMEN AND CHILD DEVELOPMENT


In [14]:
faiss.write_index(
    index,
    "parliament.index"
)

np.save(
    "question_embeddings.npy",
    embeddings
)
